# VAE for sequence learning

In sequence learning, a VAE (variational autoencoder) is useful when you want a compact latent representation of sequences rather than only a direct prediction. This latent space can capture global structure (for example, composition and motif combinations), support interpolation between sequences, and enable generation of new synthetic sequences by sampling latent vectors and decoding them. It is therefore especially helpful for unsupervised exploration, representation learning, and data augmentation workflows.

A VAE has an encoder that outputs the mean and log-variance of a latent Gaussian distribution, a reparameterization step to sample a latent vector, and a decoder that reconstructs the input. The training objective combines reconstruction loss with a KL divergence term that regularizes the latent space toward a standard normal distribution.

Steps:
```text
input x
↓
encoder
↓
mu, logvar
↓
sample z
↓
decoder
↓
reconstructed x_hat
```


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
from torch.utils.data import Dataset, DataLoader

In [2]:
# Minimal VAE in PyTorch

class SequenceVAE(nn.Module):
    def __init__(self, 
                 seq_len=7, 
                 alphabet_size=4, 
                 hidden_dim=64, 
                 latent_dim=16):
        super().__init__()
        input_dim = seq_len * alphabet_size

        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

        self.fc2 = nn.Linear(latent_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, input_dim)

        self.seq_len = seq_len
        self.alphabet_size = alphabet_size

    def encode(self, x):
        # x shape: (batch, seq_len, 4)
        x = x.view(x.size(0), -1)
        h = F.relu(self.fc1(x))
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = F.relu(self.fc2(z))
        x_hat = torch.sigmoid(self.fc3(h))
        x_hat = x_hat.view(-1, self.seq_len, self.alphabet_size)
        return x_hat

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z)
        return x_hat, mu, logvar

In [3]:
# Training loop for VAE

VOCAB = "AUCG"
def one_hot_encode(seq: str) -> torch.Tensor:
    seq = seq.upper().strip()
    idx = torch.tensor([VOCAB.index(ch) for ch in seq], 
                       dtype=torch.long)
    mat_eye = torch.eye(len(VOCAB), 
                     dtype=torch.float32)[idx]
    return(mat_eye)

# ---- toy data ----
sequences = [
    "AAAUGCC",
    "AUGCGAA",
    "UUUGGCA"
]
targets = [0.8, 0.3, 0.6]

# ---- create dataset ----
class RNADataset(Dataset):
    def __init__(self, sequences, targets):
        self.sequences = sequences
        self.targets = targets
    def __len__(self):
        return len(self.sequences)
    def __getitem__(self, idx):
        seq = self.sequences[idx]
        y = self.targets[idx]
        x = one_hot_encode(seq).T   # (4, seq_len)
        return x, torch.tensor(y, dtype=torch.float32)
dataset = RNADataset(sequences, targets)

# ---- create dataloader ----
loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True
)

# ---- create dataloader ----
model = SequenceVAE(seq_len=7, alphabet_size=4, hidden_dim=64, latent_dim=16)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
def vae_loss(x_hat, x, mu, logvar):
    recon_loss = F.binary_cross_entropy(x_hat, x, reduction="sum")
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + kl_loss

for epoch in range(50):
    model.train()
    total_loss = 0
    for x_batch, y_batch in loader:
        optimizer.zero_grad()

        x_hat, mu, logvar = model(x_batch)
        loss = vae_loss(x_hat, # x_hat -> [2, 7, 4]
                        x_batch.transpose(1, 2), # x_batch -> [2, 4, 7]
                        mu, 
                        logvar)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"epoch={epoch}, loss={total_loss:.2f}")

epoch=0, loss=59.05
epoch=1, loss=59.13
epoch=2, loss=57.68
epoch=3, loss=57.69
epoch=4, loss=56.78
epoch=5, loss=56.66
epoch=6, loss=55.49
epoch=7, loss=54.23
epoch=8, loss=55.73
epoch=9, loss=53.14
epoch=10, loss=54.45
epoch=11, loss=52.73
epoch=12, loss=53.04
epoch=13, loss=52.44
epoch=14, loss=51.82
epoch=15, loss=49.95
epoch=16, loss=47.61
epoch=17, loss=48.41
epoch=18, loss=52.34
epoch=19, loss=48.13
epoch=20, loss=49.48
epoch=21, loss=45.50
epoch=22, loss=49.46
epoch=23, loss=48.78
epoch=24, loss=46.80
epoch=25, loss=46.75
epoch=26, loss=48.25
epoch=27, loss=46.56
epoch=28, loss=43.56
epoch=29, loss=48.52
epoch=30, loss=45.51
epoch=31, loss=45.83
epoch=32, loss=46.86
epoch=33, loss=43.61
epoch=34, loss=41.94
epoch=35, loss=43.34
epoch=36, loss=41.18
epoch=37, loss=45.70
epoch=38, loss=38.29
epoch=39, loss=40.94
epoch=40, loss=38.77
epoch=41, loss=42.77
epoch=42, loss=41.33
epoch=43, loss=42.25
epoch=44, loss=40.10
epoch=45, loss=39.47
epoch=46, loss=40.33
epoch=47, loss=39.10
ep

## Supervised vs unsupervised:

- MLP/CNN/Transformer: sequence → predict a value
- VAE: sequence → (reconstructed sequence, latent representation); reconstruct + embed + generate

## Options for VAE to consider:

A trained VAE can be used to reconstruct sequences, embed sequences into a latent space via the encoder, or generate new sequences by sampling from the latent space and decoding.


### Option 1: reconstruct a sequence

In [4]:
def reconstruct_sequence(seq, model):
    x = one_hot_encode(seq).unsqueeze(0)  # (1, seq_len, 4)
    model.eval()
    with torch.no_grad():
        x_hat, mu, logvar = model(x)
    return x_hat
seq = "AAAUGCC"
x_hat = reconstruct_sequence(seq, model)
print(x_hat)

# Make it human-readable (decode back to letters)
# Convert probabilities to nucleotides:
def decode_one_hot(x_hat):
    idx_to_nt = ["A", "U", "G", "C"]
    indices = x_hat.argmax(dim=-1)  # (batch, seq_len)
    seqs = []
    for row in indices:
        seq = "".join([idx_to_nt[i] for i in row])
        seqs.append(seq)
    return seqs
decoded = decode_one_hot(x_hat)
print(decoded)

tensor([[[0.5747, 0.4687, 0.1688, 0.1378],
         [0.5506, 0.6085, 0.1821, 0.2844],
         [0.3675, 0.3696, 0.2235, 0.4648],
         [0.1800, 0.3498, 0.2567, 0.3838],
         [0.1847, 0.2152, 0.1600, 0.7357],
         [0.4351, 0.1766, 0.6379, 0.2520],
         [0.5745, 0.2882, 0.4162, 0.1373]]])
['AUCCCGA']


### Option 2: get latent representation

In [5]:
def encode_sequence(seq, model):
    x = one_hot_encode(seq).unsqueeze(0)

    model.eval()
    with torch.no_grad():
        mu, logvar = model.encode(x)

    return mu
z = encode_sequence("AAAUGCC", model)
print(z)
print(z.shape)

tensor([[ 0.0206,  0.2811,  0.2524,  0.1410, -0.3556,  0.3237,  0.0559, -0.2821,
         -0.0091, -0.0134, -0.0254,  0.0136,  0.5350,  0.1694, -0.2164, -0.1229]])
torch.Size([1, 16])


### Option 3: generate new sequences

In [ ]:
def generate_sequence(model, latent_dim):
    z = torch.randn(1, latent_dim)
    model.eval()
    with torch.no_grad():
        x_hat = model.decode(z)
    return decode_one_hot(x_hat)
new_seq = generate_sequence(model, 
                            latent_dim=16)
print(new_seq)

['AUCCCGA']
